In [10]:
import gdsfactory as gf
import numpy as np
from gdsfactory.typings import LayerSpec
from layers import LayerMap
import uuid

# ==============================================================================
# 2. Matrix Generation Function
# ==============================================================================
def generate_design_matrix(rows: int, cols: int) -> np.ndarray:
    """
    Generates a random binary design matrix (0 or 1).
    You can easily swap this function later for an optimization algorithm.
    """
    return np.random.randint(0, 2, size=(rows, cols))

# ==============================================================================
# 3. Drawing Function (FIXED)
# ==============================================================================
def draw_pixelated_device(
    design_matrix: np.ndarray,
    layer_top: LayerSpec,
    layer_ground: LayerSpec,
    pitch_x: float = 0.5,
    pitch_y: float = 0.5,
    pixel_w: float = 0.6,
    pixel_h: float = 0.6,
    tl_w: float = 2.0,
    tl_l: float = 10.0,
) -> gf.Component:
    
    # Generate unique name to prevent "Cell already exists" error
    unique_name = f"pixelated_device_{uuid.uuid4().hex[:8]}"
    c = gf.Component(unique_name)
    
    rows, cols = design_matrix.shape

    # --- Geometry Calculations ---
    # We calculate the exact dimensions mathematically to avoid querying the component
    # Grid total width (center-to-center)
    grid_width_centers = (cols - 1) * pitch_x
    grid_height_centers = (rows - 1) * pitch_y
    
    # Center offsets
    x0 = -grid_width_centers / 2
    y0 = -grid_height_centers / 2

    # --- Draw Pixels ---
    pixel_rect = gf.components.rectangle(
        size=(pixel_w, pixel_h), 
        layer=layer_top, 
        centered=True
    )

    for r in range(rows):
        for col in range(cols):
            if design_matrix[r, col] == 1:
                x_pos = x0 + (col * pitch_x)
                y_pos = y0 + (r * pitch_y)
                
                ref = c << pixel_rect
                ref.move((x_pos, y_pos))

    # --- Draw Input/Output Transmission Lines ---
    tl_rect = gf.components.rectangle(
        size=(tl_l, tl_w), 
        layer=layer_top, 
        centered=True
    )
    
    # Calculate connection points (where the grid "physically" ends for the TL)
    # We assume the TL connects to the edge of the grid pitch, similar to the image
    grid_left_edge = x0 - (pitch_x / 2)
    grid_right_edge = x0 + (cols - 1) * pitch_x + (pitch_x / 2)

    # Left TL (Port 1)
    tl1 = c << tl_rect
    tl1.movey(0)
    tl1.xmax = grid_left_edge
    
    # Right TL (Port 2)
    tl2 = c << tl_rect
    tl2.movey(0)
    tl2.xmin = grid_right_edge

    # --- Draw Ground Plane (Calculated) ---
    margin = 2.0
    
    # Calculate total width: Left TL + Grid + Right TL + Margins
    # Total physical width = (TL Length) + (Grid Width) + (TL Length)
    total_device_width = (grid_right_edge - grid_left_edge) + (2 * tl_l)
    
    # Calculate total height: Grid Height (or TL width if larger)
    # We use the larger of the pixel grid height or the TL width
    # Note: Pixel grid physical height = centers + 1 pixel width (approx)
    grid_physical_height = grid_height_centers + pixel_h
    total_device_height = max(grid_physical_height, tl_w)

    gnd_w = total_device_width + (2 * margin)
    gnd_h = total_device_height + (2 * margin)
    
    c << gf.components.rectangle(
        size=(gnd_w, gnd_h), 
        layer=layer_ground, 
        centered=True
    )

    return c

# ==============================================================================
# Execution
# ==============================================================================

# 1. Instantiate your layer map
layermap = LayerMap

# 2. Generate the design matrix independently
my_matrix = generate_design_matrix(rows=20, cols=30)

# 3. Pass the matrix and the specific layers into the drawing function
device = draw_pixelated_device(
    design_matrix=my_matrix,
    layer_top=layermap.LB,
    layer_ground=layermap.JA,
    tl_w=2.0 # Thin transmission line
)

device.show()

2026-01-12 13:57:29.780 | INFO     | kfactory.kcell:show:3995 - klive v0.4.1: Reloaded file '/tmp/3689286768.oas'
